## Split data into Train Validation and Test csv's

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Grab file
file_path = "../mid_datav1.csv"
init_df = pd.read_csv(file_path)

In [3]:
print(init_df.dtypes)

MSG_TYPE                 int64
MMSI                     int64
NAME                    object
IMO_NUMBER               int64
CALL_SIGN               object
LAT_AVG                float64
LON_AVG                float64
PERIOD                  object
SPEED_KNOTS            float64
COG_DEG                float64
HEADING_DEG            float64
NAV_STATUS               int64
NAV_SENSOR               int64
SHIP_AND_CARGO_TYPE      int64
DRAUGHT                float64
DRAUGHT.1              float64
DIM_BOW                  int64
DIM_STERN                int64
DIM_PORT                 int64
DIM_STARBOARD            int64
DESTINATION             object
MMSI_COUNTRY_CD         object
RECEIVER                object
BEAM                     int64
LENGTH                   int64
CHANNEL_SIDE            object
time_diff              float64
cog_diff               float64
new_voyage                bool
voyage_id               object
geometry                object
in_channel_north          bool
in_chann

In [4]:
# Only take columns needed for TCN Neural Network
df = init_df[['MMSI', 'LAT_AVG', 'LON_AVG', 'PERIOD', 'SPEED_KNOTS', 'COG_DEG', 'HEADING_DEG', 'voyage_id']]

# Make a datetime type column
df['TIME'] = pd.to_datetime(df['PERIOD'])
df = df.rename(columns = {'LAT_AVG':'LAT', 'LON_AVG':'LON', 'SPEED_KNOTS':'SPEED', 'COG_DEG':'COG', 'HEADING_DEG':'HEADING'})

print("datetime check", df['TIME'].dtypes)

# drop all NA's
print("pre-NA drop", len(df))
df = df.dropna()
print("post-NA drop", len(df))

datetime check datetime64[ns]
pre-NA drop 44521
post-NA drop 42883


C:\Users\LAKwon\AppData\Local\Temp\ipykernel_9140\3480178791.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['TIME'] = pd.to_datetime(df['PERIOD'])


In [5]:
# make delta time, lat, and lon columns for TCN
df['dt'] = df['TIME'].diff().dt.total_seconds()
df['dlat'] = df['LAT'].diff()
df['dlon'] = df['LON'].diff()

# sort values by vessel and time
df = df.sort_values(["MMSI", "TIME"]).reset_index(drop=True)

# fast way to change each new voyage delta time, lat, and lon from NaN to 0
mask = df['voyage_id'] != df['voyage_id'].shift()
df.loc[mask, ['dt', 'dlat', 'dlon']] = 0

# older slower way
'''for voyage in range(0, len(df)):
    if voyage > 0 and df.loc[voyage-1, "dt"] != df.loc[voyage, "dt"]:
        df.loc[voyage, ["dt", "dlat", "dlon"]] = 0'''

# change cog and heading to cos and sin for TCN
df['cog_rad'] = np.deg2rad(df['COG'])
df['cog_sin'] = np.sin(df['cog_rad'])
df['cog_cos'] = np.cos(df['cog_rad'])

df['hdg_rad'] = np.deg2rad(df['HEADING'])
df['hdg_sin'] = np.sin(df['hdg_rad'])
df['hdg_cos'] = np.cos(df['hdg_rad'])



df.head()

,MMSI,LAT,LON,PERIOD,SPEED,COG,HEADING,voyage_id,TIME,dt,dlat,dlon,cog_rad,cog_sin,cog_cos,hdg_rad,hdg_sin,hdg_cos
0,205089000,21.908479,-76.971564,3/1/2023 08:25,13.4,121.0,119.0,13_205089000,2023-03-01 08:25:00,0.0,0.000000,0.000000,2.111848,0.857167,-0.515038,2.076942,0.874620,-0.484810
1,205089000,21.901958,-76.959594,3/1/2023 08:30,13.2,116.7,111.0,13_205089000,2023-03-01 08:30:00,300.0,-0.006521,0.011970,2.036799,0.893371,-0.449319,1.937315,0.933580,-0.358368
2,205089000,21.895055,-76.938755,3/1/2023 08:35,13.2,110.2,108.0,13_205089000,2023-03-01 08:35:00,300.0,-0.006903,0.020839,1.923353,0.938493,-0.345298,1.884956,0.951057,-0.309017
3,205089000,21.876328,-76.888575,3/1/2023 08:45,13.4,112.2,109.0,13_205089000,2023-03-01 08:45:00,600.0,-0.018727,0.050180,1.958259,0.925871,-0.377841,1.902409,0.945519,-0.325568
4,205089000,21.872654,-76.878911,3/1/2023 08:50,13.4,112.0,108.0,13_205089000,2023-03-01 08:50:00,300.0,-0.003674,0.009664,1.954769,0.927184,-0.374607,1.884956,0.951057,-0.309017


In [6]:
# Reduce columns again
training_df = df[["MMSI", "voyage_id", "dt", "cog_sin", "cog_cos", "hdg_sin", "hdg_cos", "dlat", "dlon"]]

training_df.head()

,MMSI,voyage_id,dt,cog_sin,cog_cos,hdg_sin,hdg_cos,dlat,dlon
0,205089000,13_205089000,0.0,0.857167,-0.515038,0.874620,-0.484810,0.000000,0.000000
1,205089000,13_205089000,300.0,0.893371,-0.449319,0.933580,-0.358368,-0.006521,0.011970
2,205089000,13_205089000,300.0,0.938493,-0.345298,0.951057,-0.309017,-0.006903,0.020839
3,205089000,13_205089000,600.0,0.925871,-0.377841,0.945519,-0.325568,-0.018727,0.050180
4,205089000,13_205089000,300.0,0.927184,-0.374607,0.951057,-0.309017,-0.003674,0.009664


In [7]:
# List of vessels
vessels = df['MMSI'].unique()

# Shuffle for randomness (VERY important)
rng = np.random.default_rng(42)
rng.shuffle(vessels)

n_total = len(vessels)

# ~90% train and ~10% testing
test_split = int(n_total * 0.9)
train_val_vessels = vessels[:test_split]
test_vessels      = vessels[test_split:]

# Split train into ~75% train and ~15% val
val_frac_of_train = 0.2   # ~20% of the ~90%
val_split = int(len(train_val_vessels) * (1 - val_frac_of_train))

train_vessels = train_val_vessels[:val_split]
val_vessels   = train_val_vessels[val_split:]

# Create DataFrames
df_train = df[df['MMSI'].isin(train_vessels)].copy()
df_val   = df[df['MMSI'].isin(val_vessels)].copy()
df_test  = df[df['MMSI'].isin(test_vessels)].copy()

# Diagnostics
print("Train %:", len(df_train)/len(df))
print("Val %:",   len(df_val)/len(df))
print("Test %:",  len(df_test)/len(df))

print("Train rows:", len(df_train))
print("Val rows:",   len(df_val))
print("Test rows:",  len(df_test))

Train %: 0.7505305132570016
Val %: 0.17065037427418792
Test %: 0.07881911246881049
Train rows: 32185
Val rows: 7318
Test rows: 3380


In [8]:
# More Diagnostics
print("number of Train vessels:", df_train["MMSI"].nunique())
print("number of Val vessels:", df_val["MMSI"].nunique())
print("number of Test vessels:", df_test["MMSI"].nunique())

print("number of Train voyages:", df_train["voyage_id"].nunique())
print("number of Val voyages:", df_val["voyage_id"].nunique())
print("number of Test voyages:", df_test["voyage_id"].nunique())

number of Train vessels: 895
number of Val vessels: 224
number of Test vessels: 125
number of Train voyages: 2905
number of Val voyages: 691
number of Test voyages: 292


In [9]:
# Create csv's
df_train.to_csv("mid_train_data.csv", index=False)
df_val.to_csv("mid_val_data.csv", index=False)
df_test.to_csv("mid_test_data.csv", index=False)